In [ ]:
!pip install textstat pandas shap
from google.colab import drive
drive.mount('/content/drive')

## Step 1: Global Imports and Configuration
This cell imports all required data mining, NLP, and machine learning libraries. It also defines the global file paths and the constant Regex patterns used to mathematically quantify pedagogical guardrail failures (Code Leakage) and semantic alignment (C-Keywords).

In [ ]:
# ==========================================
# --- STEP 1: IMPORTS & CONFIGURATION ---
# ==========================================
import json
import glob
import os
import re
import pandas as pd
import numpy as np
import textstat
import shap
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, RocCurveDisplay, ConfusionMatrixDisplay

# --- VISUALIZATION SETUP ---
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)

# --- FILE PATHS ---
# Input directories
INPUT_JSONL_DIR = "/content/drive/MyDrive/LLM_Results/Eval"
TECH_EVAL_CSV_PATH = "/content/drive/MyDrive/LLM_Results/Eval/Phase3_Master_Evaluation_Stats.csv"

# Output directory for the new extracted features
OUTPUT_FEATURES_CSV = "/content/drive/MyDrive/Phase4_PKDD_Complete_Features.csv"

# --- NLP REGEX PATTERNS & CONSTANTS ---
# Detects C-syntax leakage in natural language Socratic hints
CODE_LEAK_PATTERN = re.compile(r'(;\s*$|->|\*ptr|malloc\(|printf\()', re.MULTILINE)

# Robustly splits Socratic hints regardless of LLM formatting hallucinations
HINT_SPLIT_PATTERN = re.compile(r'(?i)\bhint\s*#?\d*\b[:\-\.]?|^\s*\d+\.\s*', re.MULTILINE)

# Core concepts to check for Semantic Alignment in the summaries
C_KEYWORDS = ['malloc', 'free', 'struct', 'pointer', 'array', 'while', 'for', 'return', 'int', 'char', 'sizeof']

print("Step 1 Complete: Libraries imported and global configurations set.")

## Step 2: Pedagogical Feature Extraction
This script processes the raw conversational logs (`.jsonl` files) to extract 12 distinct pedagogical and structural features. It quantifies "Extraneous Cognitive Load" via readability formulas (Flesch-Kincaid, Gunning Fog), assesses "Semantic Alignment" by calculating the keyword intersection between the generated C-code and the summary, and measures "Pedagogical Dissonance" by analyzing the escalation in complexity across Socratic hints.

In [ ]:
# ==========================================
# --- STEP 2: FEATURE EXTRACTION ENGINE ---
# ==========================================
print("Starting Feature Extraction across JSONL dataset...")

jsonl_files = glob.glob(os.path.join(INPUT_JSONL_DIR, "*.jsonl"))
extracted_data = []

for file_path in jsonl_files:
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                data = json.loads(line)
                steps = data.get("steps", {})
                if not steps: continue

                iteration = data.get("iteration")
                model = data.get("model")
                topic = data.get("topic")

                # --- STEP 2: CODE STRUCTURE ---
                step_2_code = steps.get("step_2", "")
                code_lines = step_2_code.split('\n')
                comment_lines = sum(1 for line in code_lines if line.strip().startswith('//') or line.strip().startswith('/*'))
                total_lines = len(code_lines) if len(code_lines) > 0 else 1
                comment_ratio = round(comment_lines / total_lines, 3)

                # --- STEP 3: EXPLANATION COMPLEXITY ---
                step_3_text = steps.get("step_3", "")
                exp_fk_grade = textstat.flesch_kincaid_grade(step_3_text)
                exp_fog = textstat.gunning_fog(step_3_text)

                words_exp = re.findall(r'\w+', step_3_text.lower())
                exp_word_count = len(words_exp)
                exp_lex_div = round(len(set(words_exp)) / exp_word_count, 3) if exp_word_count > 0 else 0

                # --- STEP 4: SCAFFOLDING ---
                step_4_hints = steps.get("step_4", "")
                code_leakage_flag = 1 if CODE_LEAK_PATTERN.search(step_4_hints) else 0

                hint_blocks = HINT_SPLIT_PATTERN.split(step_4_hints)
                hint_blocks = [b.strip() for b in hint_blocks if len(b.strip()) > 10]

                hint_progression = 0
                if len(hint_blocks) >= 2:
                    hint_progression = textstat.flesch_kincaid_grade(hint_blocks[-1]) - textstat.flesch_kincaid_grade(hint_blocks[0])

                # --- STEP 5: SUMMARY & ALIGNMENT ---
                step_5_summary = steps.get("step_5", "")
                sum_fk_grade = textstat.flesch_kincaid_grade(step_5_summary) if step_5_summary else 0
                sum_word_count = len(re.findall(r'\w+', step_5_summary.lower())) if step_5_summary else 0

                # Semantic Alignment Check
                code_lower = step_2_code.lower()
                sum_lower = step_5_summary.lower()
                used_concepts = [c for c in C_KEYWORDS if c in code_lower]

                if used_concepts and step_5_summary:
                    overlap = sum(1 for c in used_concepts if c in sum_lower)
                    sum_alignment = round(overlap / len(used_concepts), 3)
                else:
                    sum_alignment = 0

                # --- STEP 6: CONSTRAINTS & FORMATTING ---
                step_6_tests = steps.get("step_6", "")
                constraint_memory = 1 if "exit" in step_6_tests.lower() else 0

                json_valid = 1
                if "{" in step_6_tests:
                    try:
                        json_str = step_6_tests[step_6_tests.find("{"):step_6_tests.rfind("}")+1]
                        json.loads(json_str)
                    except:
                        json_valid = 0
                else:
                    json_valid = 0

                # Append Row
                extracted_data.append({
                    "Iteration": iteration,
                    "Model": model,
                    "Topic": topic,
                    "Comment_Ratio": comment_ratio,
                    "Exp_FK_Grade": exp_fk_grade,
                    "Exp_Gunning_Fog": exp_fog,
                    "Exp_Lex_Diversity": exp_lex_div,
                    "Exp_Word_Count": exp_word_count,
                    "Hint_Code_Leakage": code_leakage_flag,
                    "Hint_Progression_Delta": round(hint_progression, 2),
                    "Sum_Alignment_Score": sum_alignment,
                    "Sum_FK_Grade": sum_fk_grade,
                    "Sum_Word_Count": sum_word_count,
                    "Test_JSON_Valid": json_valid,
                    "Constraint_Memory": constraint_memory
                })

            except Exception as e:
                continue

# --- SAVE & VERIFY ---
df_features = pd.DataFrame(extracted_data)
df_features.to_csv(OUTPUT_FEATURES_CSV, index=False)

print("\n" + "="*50)
print("EXTRACTION VERIFICATION REPORT")
print("="*50)
print(f"Total Iterations Processed : {len(df_features)}")
print(f"Extracted Deep Features    : {len(df_features.columns) - 3}")
print(f"File Saved To              : {OUTPUT_FEATURES_CSV}")
print("-" * 50)
print("Preview of Data Sample (First 5 Rows):")
display(df_features[['Model', 'Exp_FK_Grade', 'Hint_Code_Leakage', 'Sum_Alignment_Score']].head())

## Step 3: Data Harmonization and Merging
This cell acts as the data engineering bridge. Because the pedagogical features and the technical execution stats (from Phase 3) were generated by different pipelines, their identifier strings (`Model` and `Topic`) do not perfectly align. This script cleans the API paths, harmonizes the topic strings, and performs an inner join to create the final unified matrix. It also defines the target variable for the Machine Learning model: `Technical_Failure` (defined as a Memory Safety Score of 0).

In [ ]:
# ==========================================
# --- STEP 3: DATA CLEANING & MERGING ---
# ==========================================
print("Starting Data Harmonization...")

# 1. Load the Datasets
df_features = pd.read_csv(OUTPUT_FEATURES_CSV)
df_tech = pd.read_csv(TECH_EVAL_CSV_PATH)

# ==========================================
# STRING HARMONIZATION
# ==========================================
# Fix Models: 'openai/gpt-oss-120b' -> 'openai'
df_features['Model'] = df_features['Model'].apply(lambda x: x.split('/')[0] if '/' in x else x)

# Fix Features Topics: Replace underscores with spaces
df_features['Topic'] = df_features['Topic'].str.replace('_', ' ')

# Fix Tech Topics: Strip out the accidental model prefixes from Phase 3
valid_topics = [
    'Pointers and Pointer Arithmetic',
    'Dynamic Memory Allocation (malloc, free)',
    'Implementing Data Structures (e.g., Singly Linked Lists)'
]
def clean_topic_string(text):
    for vt in valid_topics:
        if vt in text:
            return vt
    return text

df_tech['Topic'] = df_tech['Topic'].apply(clean_topic_string)

# ==========================================
# MERGING & TARGET VARIABLE CREATION
# ==========================================
# Inner join drops any rows that failed to compile in Phase 3
df_merged = pd.merge(df_features, df_tech, on=["Iteration", "Model", "Topic"], how="inner")

# Define Target: 1 if it has a memory leak (Mem_Safety_Score == 0), 0 if it is safe
df_merged['Technical_Failure'] = (df_merged['Mem_Safety_Score'] == 0).astype(int)

print("\n" + "="*50)
print("MERGE VERIFICATION REPORT")
print("="*50)
print(f"Final Merged Dataset Size : {len(df_merged)} rows")
print(f"Target Variable Created   : 'Technical_Failure'")
print("-" * 50)
# Show the balance of the dataset (how many failed vs succeeded)
failure_counts = df_merged['Technical_Failure'].value_counts()
print(f"Class Balance (0 = Safe, 1 = Leak):")
print(f"  Safe Codes (0) : {failure_counts.get(0, 0)}")
print(f"  Memory Leaks (1): {failure_counts.get(1, 0)}")
print("="*50)

## Step 4: Predictive Modeling (Random Forest)
This cell trains a Random Forest Classifier to predict technical execution failures (Memory Leaks) using exclusively the natural language pedagogical features extracted in Step 2. As outlined in the methodology, Random Forest is utilized for its robustness on tabular data and its ability to handle non-linear feature interactions. The model's efficacy is verified using a Receiver Operating Characteristic (ROC) Curve and a Confusion Matrix.

In [ ]:
# ==========================================
# --- STEP 4: PREDICTIVE MODELING (RANDOM FOREST) ---
# ==========================================
print("Training Random Forest Classifier...")

# 1. Define Features (X) and Target (y)
FEATURE_COLS = [
    "Comment_Ratio",
    "Exp_FK_Grade",
    "Exp_Gunning_Fog",
    "Exp_Lex_Diversity",
    "Exp_Word_Count",
    "Hint_Code_Leakage",
    "Hint_Progression_Delta",
    "Sum_Alignment_Score",
    "Sum_FK_Grade",
    "Sum_Word_Count",
    "Test_JSON_Valid",
    "Constraint_Memory"
]

X = df_merged[FEATURE_COLS]
y = df_merged['Technical_Failure']

# 2. Train/Test Split (80/20 with stratification to maintain the 685/510 ratio)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 3. Train the Model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced", max_depth=10)
rf_model.fit(X_train, y_train)

# 4. Evaluate Predictions
y_pred = rf_model.predict(X_test)
y_proba = rf_model.predict_proba(X_test)[:, 1]
auc_score = roc_auc_score(y_test, y_proba)

print("\n" + "="*50)
print("RANDOM FOREST PREDICTION RESULTS")
print("="*50)
print(f"ROC-AUC Score : {auc_score:.3f}")
print("-" * 50)
print(classification_report(y_test, y_pred, target_names=["Safe (0)", "Memory Leak (1)"]))

# 5. Generate ROC & Confusion Matrix
fig, ax = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: ROC Curve
RocCurveDisplay.from_estimator(rf_model, X_test, y_test, ax=ax[0], color='darkorange', linewidth=2)
ax[0].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
ax[0].set_title('Receiver Operating Characteristic (ROC) Curve', fontweight='bold')
ax[0].set_xlabel('False Positive Rate')
ax[0].set_ylabel('True Positive Rate')

# Plot 2: Confusion Matrix
cm_disp = ConfusionMatrixDisplay.from_estimator(rf_model, X_test, y_test, ax=ax[1], cmap='Blues', colorbar=False, display_labels=["Safe (0)", "Leak (1)"])
for text in cm_disp.text_.ravel():
    text.set_fontsize(24)
    text.set_fontweight('bold')
ax[1].set_title('Random Forest Confusion Matrix', fontweight='bold')
ax[1].grid(False)

plt.tight_layout()
plt.savefig("Figure_ML_Evaluation_Metrics.png", dpi=300)
plt.show()
print("Figure Saved: 'Figure_ML_Evaluation_Metrics.png'")

## Step 5: Explainable AI (SHAP Value Analysis)
To transition from predictive modeling to true Knowledge Discovery, this cell applies SHAP (SHapley Additive exPlanations). Rooted in cooperative game theory, SHAP calculates the marginal contribution of each linguistic feature to the final prediction. The resulting Summary Plot breaks open the "black box" of the Random Forest, visually proving how metrics like Extraneous Cognitive Load (FK Grade) and Pedagogical Dissonance (Hint Delta) dynamically push the model toward predicting a structural code failure.

In [ ]:
# ==========================================
# --- STEP 5: KNOWLEDGE DISCOVERY (SHAP) ---
# ==========================================
print("Generating SHAP Explanations (This may take a few seconds)...")

# 1. Initialize the TreeExplainer
explainer = shap.TreeExplainer(rf_model)

# 2. Calculate SHAP values for the test set
shap_values = explainer.shap_values(X_test)

# 3. Extract the SHAP values for the positive class (Technical_Failure = 1)
if isinstance(shap_values, list):
    shap_values_to_plot = shap_values[1]
else:
    if len(shap_values.shape) == 3:
        shap_values_to_plot = shap_values[:,:,1]
    else:
        shap_values_to_plot = shap_values

# 4. Generate the SHAP Summary Plot
plt.figure(figsize=(12, 8))
plt.title("SHAP Summary Plot: Predicting C Memory Leaks via Linguistic Decay", fontsize=14, fontweight='bold')

shap.summary_plot(shap_values_to_plot, X_test, plot_type="dot", show=False)

plt.tight_layout()
plt.savefig("Figure_SHAP_Summary.png", dpi=300, bbox_inches='tight')
plt.show()

print("Figure Saved: 'Figure_SHAP_Summary.png'")

## Step 6: Pedagogical & Architectural Analysis (RQ2 & RQ3)
This final cell generates the empirical visualizations and descriptive statistics required to directly answer the core research questions. It produces a boxplot to measure "Pedagogical Dissonance" (how much the models unnaturally escalate their hint complexity), a bar chart to quantify "Architectural Degradation" (failure rates in retaining constraints and JSON formatting), and a violin plot to assess "Semantic Decoupling" in the generated summaries. Finally, it exports Table 4 containing the mean and standard deviation of all key features.

In [ ]:
# ==========================================
# --- STEP 6: PEDAGOGICAL & ARCHITECTURAL ANALYSIS ---
# ==========================================
print("Generating Empirical Visualizations for RQ2 & RQ3...")

# ------------------------------------------
# Plot 1: Pedagogical Dissonance (RQ3)
# ------------------------------------------
plt.figure(figsize=(10, 6))
sns.boxplot(
    data=df_merged,
    x='Model',
    y='Hint_Progression_Delta',
    hue='Model',
    palette="Set2",
    legend=False,
    showmeans=True,
    meanprops={"marker":"o", "markerfacecolor":"white", "markeredgecolor":"black"}
)
plt.title("Pedagogical Dissonance: Escalation in Hint Complexity (RQ3)", fontsize=14, fontweight='bold')
plt.xlabel("Frontier Model", fontsize=12)
plt.ylabel("FK Grade Delta (Hint 3 - Hint 1)", fontsize=12)
plt.axhline(0, color='red', linestyle='--', linewidth=1.5, alpha=0.7)
plt.tight_layout()
plt.savefig("Figure_Pedagogical_Dissonance.png", dpi=300)
plt.show()

# ------------------------------------------
# Plot 2: Context Drift & Architectural Degradation (RQ2)
# ------------------------------------------
# Calculate failure rates (1.0 - mean success rate) and convert to percentage
degradation_data = df_merged.groupby('Model')[['Test_JSON_Valid', 'Constraint_Memory']].mean()
degradation_data = (1.0 - degradation_data) * 100

ax = degradation_data.plot(kind='bar', figsize=(10, 6), color=['#e74c3c', '#3498db'], edgecolor='black')
plt.title("Architectural Degradation: Memory & Formatting Failure Rates (RQ2)", fontsize=14, fontweight='bold')
plt.xlabel("Frontier Model", fontsize=12)
plt.ylabel("Failure Rate (%)", fontsize=12)
plt.xticks(rotation=0)
plt.legend(["JSON Formatting Breakdown", "Forgot 'EXIT' Constraint"], title="Failure Type")

# Add percentage labels above bars
for p in ax.patches:
    ax.annotate(f"{p.get_height():.1f}%",
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', xytext=(0, 5), textcoords='offset points')

plt.tight_layout()
plt.savefig("Figure_Architectural_Degradation.png", dpi=300)
plt.show()

# ------------------------------------------
# Plot 3: Semantic Alignment (Dissonance Audit)
# ------------------------------------------
plt.figure(figsize=(10, 6))
sns.violinplot(
    data=df_merged,
    x='Model',
    y='Sum_Alignment_Score',
    hue='Model',
    palette="Pastel1",
    legend=False,
    inner="quartile"
)
plt.title("Semantic Decoupling: Keyword Alignment in Learning Summaries", fontsize=14, fontweight='bold')
plt.xlabel("Frontier Model", fontsize=12)
plt.ylabel("Alignment Score (% of C-keywords retained)", fontsize=12)
plt.tight_layout()
plt.savefig("Figure_Semantic_Alignment.png", dpi=300)
plt.show()

# ------------------------------------------
# Table 4: Descriptive Statistics Export
# ------------------------------------------
print("\nGenerating Table 4: Descriptive Statistics (Mean ± STD)")
table_features = [
    'Exp_FK_Grade', 'Comment_Ratio', 'Hint_Progression_Delta',
    'Sum_Alignment_Score', 'Exp_Word_Count'
]

desc_stats = df_merged.groupby('Model')[table_features].agg(['mean', 'std'])
formatted_table = pd.DataFrame()

# Combine mean and std into a single string column
for col in table_features:
    formatted_table[col] = desc_stats[col].apply(
        lambda x: f"{x['mean']:.2f} ± {x['std']:.2f}", axis=1
    )

display(formatted_table)
formatted_table.to_csv("Table_Descriptive_Stats.csv")
print("All pedagogical charts generated. Table saved as 'Table_Descriptive_Stats.csv'")

## Step 7: Methodological Rigor & Statistical Proofs
To ensure conference-grade rigor, this cell validates the machine learning pipeline and the empirical findings. First, it generates a Feature Correlation Heatmap to check for multicollinearity among the linguistic variables. Second, it executes a 5-Fold Stratified Cross-Validation to prove the Random Forest's predictive stability. Finally, it calculates the Kruskal-Wallis H-test to determine if the architectural differences in 'Pedagogical Dissonance' (RQ3) are statistically significant.

In [ ]:
# ==========================================
# --- STEP 7: METHODOLOGICAL RIGOR & STATS ---
# ==========================================
from sklearn.model_selection import cross_val_score, StratifiedKFold
import scipy.stats as stats

print("Running Reviewer-Proofing Diagnostics...\n")

# ------------------------------------------
# 1. Feature Correlation Heatmap
# ------------------------------------------
plt.figure(figsize=(10, 8))
# Calculate Pearson correlation matrix
corr_matrix = df_merged[FEATURE_COLS].corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap='coolwarm',
            vmax=1, vmin=-1, square=True, linewidths=.5, cbar_kws={"shrink": .75},
            annot_kws={"size": 8})
plt.title("Feature Collinearity: Spearman Correlation Matrix", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("Figure_Correlation_Heatmap.png", dpi=300)
plt.show()

# ------------------------------------------
# 2. 5-Fold Stratified Cross-Validation
# ------------------------------------------
print("Running 5-Fold Stratified Cross-Validation...")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Calculate cross-validated ROC-AUC
cv_scores = cross_val_score(rf_model, X, y, cv=cv, scoring='roc_auc')

print(f"Cross-Validation ROC-AUC Scores : {cv_scores}")
print(f"Mean CV ROC-AUC                 : {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
if cv_scores.std() < 0.05:
    print("   -> Model performance is highly stable across data splits.")
else:
    print("   -> High variance detected in model splits.")

# ------------------------------------------
# 3. Statistical Significance Testing (RQ3)
# ------------------------------------------
print("\nRunning Statistical Significance Tests (Kruskal-Wallis)...")

# We want to see if the 'Hint_Progression_Delta' is statistically different across the 4 models
models = df_merged['Model'].unique()
hint_groups = [df_merged[df_merged['Model'] == m]['Hint_Progression_Delta'].dropna() for m in models]

# Kruskal-Wallis H-test (non-parametric ANOVA)
h_stat, p_value = stats.kruskal(*hint_groups)

print(f"RQ3: Pedagogical Dissonance Differences Across Models")
print(f"   -> H-Statistic : {h_stat:.3f}")
print(f"   -> p-value     : {p_value:.3e}")

if p_value < 0.05:
    print("   -> SIGNIFICANT: We can mathematically claim the architectures behave differently.")
else:
    print("   -> NOT SIGNIFICANT: Issue of differences (e.g., random chance)")
print("="*50)